# 00 · Setup

Everything you need to follow along, on your laptop or in the browser. Work top to bottom: get
LangSmith access, pick where to run the notebooks, fill in `.env`, then run the checks at the end.

You'll need a laptop and a browser. Use **Chrome** for LangSmith Studio.

## Step 1 · Create a new LangSmith organization

The workshop gives you temporary LangSmith access through **Workshop Studio**. It only activates
on an organization created **today or yesterday**, so make a fresh one even if you already use
LangSmith.

1. Go to [smith.langchain.com](https://smith.langchain.com) and sign up or log in.
2. Open **Settings**, click the organization dropdown, and choose **Create organization**.
3. Give it a name (for example `kea-workshop`) and finish setup.
4. Select the new organization in the dropdown, then click **ID** next to *Organization* to copy
   its **Organization ID**.

<img src="../assets/screenshots/p0-langsmith-create-org.png" alt="Settings: the Organization ID button and the Create organization option in the organization dropdown" width="280">

## Step 2 · Activate workshop access

1. Open [workshop.langchain.com](https://workshop.langchain.com/).
2. Paste your **Organization ID** and the **workshop code** we share in the room.
3. Click **Activate** and keep the page open until it says **Access activated**.
4. Refresh LangSmith. The workshop plan can take up to two minutes to appear.

<img src="../assets/screenshots/p0-workshop-studio-activate.png" alt="Workshop Studio: paste the organization ID and workshop code, then Activate" width="900">

If it says *Organization not found*, check you copied the ID of the **new** organization, wait a
minute, and try once more. *Already active* means you're done.

## Step 3 · Create an API key

In the new organization, open **Settings → API Keys → Create API key**.

- **Key Type:** Personal Access Token
- **Default Workspace:** the workspace in your new organization
- **Expiration:** anything after today

Copy the key right away. LangSmith shows it once, and it starts with `lsv2_pt_`.

<img src="../assets/screenshots/p0-langsmith-api-key.png" alt="The Create an API Key dialog" width="700">

## Step 4 · Pick where to run

| | Local (recommended) | Binder (browser only) |
|---|---|---|
| Setup | ~5 minutes | One click, ~2 minutes to start |
| Studio | ✅ | ❌ Binder can't expose the Agent Server to Studio |
| Traces, datasets, experiments | ✅ | ✅ |
| Keeps your work | ✅ | ❌ Sessions end after ~10 idle minutes |

### Local

You need [mise](https://mise.jdx.dev/getting-started.html) (it installs `uv` and the other
tools) and [VS Code](https://code.visualstudio.com/) with the Jupyter extension, or JupyterLab.

```bash
git clone https://github.com/darieldatoon/agent-engineering-workshop.git
cd agent-engineering-workshop
mise trust && mise install && mise run setup   # Python, dependencies, git hooks
cp .env.example .env                           # then fill it in (Step 5)
mise run dev                                   # Agent Server for Studio; leave this terminal open
```

Open the repo folder in VS Code and open this notebook. The first time you open each notebook,
pick the `.venv` kernel (**Select Kernel → Python Environments → .venv**); VS Code remembers it.
Prefer JupyterLab? Run `mise run lab`.

No mise? `uv sync` then `uv run langgraph dev` does the same.

### Binder

[![Launch on Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/darieldatoon/agent-engineering-workshop/main?urlpath=lab/tree/notebooks/00_setup.ipynb)

The setup cell asks for your LangSmith key, then for a gateway key, the first time, and writes
them to `.env` inside the session. Press Enter at the gateway prompt if you don't have one. Skip the Studio steps; every result still shows up in LangSmith.

## Step 5 · Fill in `.env`

Only `LANGSMITH_API_KEY` is required.

| Variable | Required? | What it's for |
|---|---|---|
| `LANGSMITH_API_KEY` | **Yes** | Your key from Step 3. Tracing, datasets, and evals use it. |
| `LANGSMITH_GATEWAY_API_KEY` | Optional | A key for model calls through the LangSmith LLM gateway, such as the shared key we give out in the room. Blank, model calls use `LANGSMITH_API_KEY`. |
| `LANGSMITH_TRACING` | Set to `true` | Sends traces to LangSmith. |
| `LANGSMITH_WORKSPACE_ID` | Optional | Only if your key reaches more than one workspace. |
| `KEA_MODEL`, `KEA_ALT_MODEL`, `KEA_JUDGE_MODEL` | Optional | Swap the models. Defaults are gateway models (`langsmith:openai/...`). To use your own provider, set for example `KEA_MODEL=openai:gpt-5.4-mini` plus `OPENAI_API_KEY`. |
| `KEA_ENABLE_MCP` | Optional | `false` turns off the web-search MCP server in Part 2 (for networks that block it). |
| `KEA_DEPLOYMENT_URL` | Part 3 | The deployed agent's URL. We share it in the room, or you deploy your own in Part 3. |

```bash
LANGSMITH_API_KEY=lsv2_pt_...
LANGSMITH_GATEWAY_API_KEY=...   # optional
LANGSMITH_TRACING=true
```

Never commit `.env`. It's in `.gitignore`.

**Typed a wrong key?** Fix the line in `.env`. On Binder, open **File → New → Terminal** and run
`rm .env` instead. Then **Kernel → Restart Kernel** and rerun the setup cell; on Binder it asks
again. If `mise run dev` is running, restart it too, since it reads `.env` only when it starts.

## Step 6 · Run the checks

Run every cell below. Each prints ✅ or ❌ with what to do next.

In [ ]:
import os


def report(ok: bool, message: str) -> None:
    print(("✅ " if ok else "❌ ") + message)


def short(exc: Exception) -> str:
    return f"{type(exc).__name__}: {str(exc).splitlines()[0][:120] if str(exc) else ''}"


ON_BINDER = any(name.startswith("BINDER_") for name in os.environ)
print("Running on", "Binder" if ON_BINDER else "your machine")

### LangSmith API key

In [ ]:
try:
    from kea.config import MODEL, setup

    setup("kea-claims-local")
    report(bool(os.environ.get("LANGSMITH_API_KEY")), "LANGSMITH_API_KEY is set")
    gateway = (
        "LANGSMITH_GATEWAY_API_KEY"
        if os.environ.get("LANGSMITH_GATEWAY_API_KEY")
        else "LANGSMITH_API_KEY"
    )
    print(f"➡️  Model calls use {gateway}")
except Exception as exc:
    report(
        False,
        f"Could not load the key ({short(exc)}). Add LANGSMITH_API_KEY to .env "
        "and pick the project's Python kernel.",
    )

### A model call through the LangSmith gateway

In [ ]:
try:
    from langchain.chat_models import init_chat_model

    reply = str(init_chat_model(MODEL).invoke("Reply with one word: ready").text)
    report(True, f"Model call worked ({MODEL}): {reply.strip()[:40]!r}")
except Exception as exc:
    report(
        False,
        f"Model call failed ({short(exc)}). Check LANGSMITH_GATEWAY_API_KEY, or that workshop "
        "access is active (Step 2). Or set KEA_MODEL to your own provider (Step 5).",
    )

### LangSmith is reachable

In [ ]:
try:
    from langsmith import Client

    list(Client().list_projects(limit=1))
    report(True, "LangSmith accepted the API key")
except Exception as exc:
    report(False, f"Could not reach LangSmith ({short(exc)}). Check the key and network.")

### The local Agent Server (for Studio)

In [ ]:
from urllib.request import urlopen

if ON_BINDER:
    print("⏭️  Skipped on Binder: Studio needs the Agent Server on your own machine.")
else:
    try:
        with urlopen("http://127.0.0.1:2024/ok", timeout=3) as response:
            report(response.status == 200, "langgraph dev is running on port 2024")
    except Exception:
        report(
            False,
            "langgraph dev is not running. Open a terminal in the repo folder and run "
            "`mise run dev`, then rerun this cell.",
        )

### Open Studio

With the server running, open
[LangSmith Studio](https://smith.langchain.com/studio/?baseUrl=http://127.0.0.1:2024) in Chrome
and click **Allow** when it asks for *Local network access*. You should see the `claims` and
`rebooking` graphs. Safari and Brave block this; use Chrome, or run `uv run langgraph dev --tunnel`.

<img src="../assets/screenshots/p1-03-studio-graph.png" alt="LangSmith Studio connected to the local Agent Server" width="900">

## Starting over

`mise run reset` clears local Studio threads and memory so you can rerun any notebook from a
clean slate. It doesn't touch LangSmith.

---

All ✅? Start with [Part 1.1 · The Agent Development Lifecycle](part1-langgraph/01_adlc.ipynb).